# Introduction to TensorFlow & Keras — Part 3
### Training a digit classifier, with experiment tracking

Everything from Parts 1 and 2, put together, with one addition: this
time every model you train gets logged to **Weights & Biases**, the
same way your classical ML models were in the Formative. You'll write
one reusable `log_experiment()` function and reuse it for every
model below.

### Code rules for this notebook
1. Keep all your imports in the next cell.
2. Every model you train must go through `log_experiment()` — don't write a second version of it.
3. Give variables names that say what they hold.
4. Add type hints to every function you write.

In [4]:
!pip install wandb

  Attempting uninstall: protobuf
    Found existing installation: protobuf 7.36.1
    Uninstalling protobuf-7.36.1:
      Successfully uninstalled protobuf-7.36.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-firestore 2.19.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.71.2 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Any, Dict

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

RANDOM_SEED = 42
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## Step 1: Load the data

In [6]:
(X_train_raw, y_train), (X_test_raw, y_test) = keras.datasets.mnist.load_data()

print(f"Training images: {X_train_raw.shape}")
print(f"Test images: {X_test_raw.shape}")
print(f"Pixel range: [{X_train_raw.min()}, {X_train_raw.max()}]")

Training images: (60000, 28, 28)
Test images: (10000, 28, 28)
Pixel range: [0, 255]


## Step 2: Normalize

MNIST pixels are always integers 0-255, so there's no guesswork here
-- divide by 255 and flatten each image, same as Parts 1 and 2.

In [7]:
X_train = X_train_raw.reshape(-1, 28 * 28).astype("float32") / 255.0
X_test = X_test_raw.reshape(-1, 28 * 28).astype("float32") / 255.0

print(f"X_train range after normalizing: [{X_train.min()}, {X_train.max()}]")

X_train range after normalizing: [0.0, 1.0]


## Step 3: Experiment tracking setup (Weights & Biases)

Set this up **before** training anything, so your baseline run below
is logged like every run after it.

Store your API key as a Colab secret -- never hardcode it in this
notebook: left sidebar -> key icon -> add secret named `WANDB_API_KEY`.

This cell is written defensively: if W&B isn't set up yet, the rest
of the notebook still runs -- you just won't get tracking until you
fix it.

In [10]:
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

load_dotenv()

WANDB_ENABLED = True
WANDB_PROJECT = "intro-tensorflow-BodeMurairi"  # <-- change this

try:
    import wandb
    from wandb.integration.keras import WandbMetricsLogger
    logged_in = True

    # --- Uncomment once your WANDB_API_KEY secret is set, then re-run this cell ---

    # from google.colab import userdata
    # wandb.login(key=userdata.get("WANDB_API_KEY"))
    wandb.login(key=os.getenv("WANDB_API_KEY"))
    logged_in = True

    WANDB_ENABLED = logged_in
    if WANDB_ENABLED:
        print("W&B ready. Runs will be logged to project:", WANDB_PROJECT)
    else:
        print("W&B installed but not logged in yet -- uncomment the block above.")
        print("Until then, the notebook still runs, but NOTHING is being tracked.")
except Exception as e:
    print("W&B not available -- runs will NOT be logged until you fix this.")
    print("Reason:", e)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/bode-murairi/.netrc


W&B ready. Runs will be logged to project: intro-tensorflow-BodeMurairi


## Step 4: One reusable function — `log_experiment()`

Same idea as `train_and_evaluate()` from Parts 1 and 2, extended to
also log to W&B. The important addition is `WandbMetricsLogger()`,
passed straight into `model.fit(...)` as a callback — it logs
**train loss, train accuracy, validation loss, and validation
accuracy at the end of every single epoch**, live, while training is
still running. That's the actual training loop, not just a summary
of how it ended.

On top of that per-epoch stream, we also log one summary (final
metrics + a learning-curve plot) once training finishes, and keep a
local record for the results table in Step 7. Every model from here
on goes through this one function.

📖 [WandbMetricsLogger docs](https://docs.wandb.ai/guides/integrations/keras/#track-experiments-with-wandbmetricslogger)

In [18]:
RANDOM_STATE = 42
results_log = []  # every experiment's summary lands here -> becomes your results table in Step 7

def log_experiment(
    run_name: str,
    model: keras.Model,
    config: Dict[str, Any],
    X_train: np.ndarray,
    y_train: np.ndarray,
    epochs: int = 30,
    batch_size: int = 128,
) -> Any:
    # Assumes `model` is already compiled. Trains with a 10% validation
    # split and early stopping, logs every epoch to W&B, and records a
    # results_log row at the end.
    #
    # run_name : short descriptive string, e.g. "dense64_dropout0.2"
    # config   : dict of whatever you want tracked, e.g. {"architecture": "Sequential", "dropout": 0.2}
    early_stopping = keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=10, restore_best_weights=True
    )

    callbacks = [early_stopping]
    run = None
    if WANDB_ENABLED:
        try:
            # wandb.init() has to happen BEFORE fit() -- WandbMetricsLogger
            # attaches to whichever run is currently active.
            run = wandb.init(project=WANDB_PROJECT, name=run_name, config=config, reinit=True)
            callbacks.append(WandbMetricsLogger())  # logs loss/accuracy + val_loss/val_accuracy every epoch
        except Exception as e:
            print(f"  (W&B run could not start: {e} -- continuing without live logging)")
            run = None

    history = model.fit(
        X_train, y_train,
        validation_split=0.1, epochs=epochs, batch_size=batch_size,
        callbacks=callbacks, verbose=0,
    )

    final_train_acc = history.history["accuracy"][-1]
    final_val_acc = history.history["val_accuracy"][-1]
    print(f"[{run_name}] train acc = {final_train_acc:.4f}, val acc = {final_val_acc:.4f}")

    if run is not None:
        try:
            fig, ax = plt.subplots(figsize=(5, 4))
            ax.plot(history.history["accuracy"], label="train")
            ax.plot(history.history["val_accuracy"], label="val")
            ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy"); ax.set_title(run_name); ax.legend()

            wandb.log({
                "final_train_accuracy": final_train_acc,
                "final_val_accuracy": final_val_acc,
                "epochs_run": len(history.history["loss"]),
                "learning_curve": wandb.Image(fig),
            })
            plt.close(fig)
        except Exception as e:
            print(f"  (W&B summary logging failed: {e})")
        finally:
            run.finish()

    results_log.append({
        "run_name": run_name,
        **config,
        "final_train_accuracy": round(final_train_acc, 4),
        "final_val_accuracy": round(final_val_acc, 4),
    })
    return model, history

## Step 5: Baseline model

A small, unremarkable Sequential model -- fully worked, no
`<FILL_IN>` -- so you have a working, fully-tracked pipeline confirmed
end-to-end, and a concrete number to beat before building anything
fancier.

Once this cell finishes, open the run link W&B prints and check the
**charts tab** — you should see train/val loss and train/val accuracy
plotted epoch by epoch, updated live while it trained, not just a
single final number.

In [13]:
baseline_model = keras.Sequential([
    layers.Dense(64, activation="relu", input_shape=(784,)),
    layers.Dense(10, activation="softmax"),
])
baseline_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

baseline_model, baseline_history = log_experiment(
    run_name="baseline-dense64",
    model=baseline_model,
    config={"architecture": "Sequential", "hidden_units": 64, "dropout": None},
    X_train=X_train, y_train=y_train,
)

E0000 00:00:1789657771.713225  188011 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1789657771.714983  194853 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1789657771.744920  188011 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'fini

[baseline-dense64] train acc = 0.9956, val acc = 0.9735


epoch/accuracy,▁▄▅▆▆▆▇▇▇▇▇▇████████
epoch/epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▄▅▆▇▇▇▇▇▇██████████
epoch/val_loss,█▅▄▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
epochs_run,▁
final_train_accuracy,▁
final_val_accuracy,▁
epoch/accuracy,0.99559
epoch/epoch,19


## Step 6: Your models

Your turn, no scaffolding: build at least two more models that beat
the baseline, using anything from Parts 1-2 -- Sequential or
Functional, Dropout, different widths or depths, more epochs. Reuse
`log_experiment(...)` for every one, with a descriptive `run_name`
and a `config` dict describing what's different about that run.

In [22]:
# Your models go here. Reuse the pattern from Step 5, e.g.:
#
my_model = keras.Sequential([
    layers.Dense(128, activation="relu", input_shape=(784,), name="first_hidden_layer"),
    layers.Dense(10, activation="softmax", name="output_hidden_layer")
    ])
my_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
my_model, my_history = log_experiment(
     run_name="sequential_model_with_more_neurones_on_the_hidden_layers",
     model=my_model,
     config={"architecture": "Sequential", "description":"Increase the number of neurones in hidden layers to check if model can extract more pattern"},
     X_train=X_train, y_train=y_train,
 )

[sequential_model_with_more_neurones_on_the_hidden_layers] train acc = 1.0000, val acc = 0.9797


epoch/accuracy,▁▅▆▆▇▇▇▇▇▇████████████████████
epoch/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▃▄▅▆▆▆▆▆▇▇▇▇▇▇▇████▇▇▇▇▇█▇▇██
epoch/val_loss,█▅▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▂▃▃▃▄▃▃▄▃▃▃
epochs_run,▁
final_train_accuracy,▁
final_val_accuracy,▁
epoch/accuracy,0.99996
epoch/epoch,29


In [ ]:
model_2 = keras.F

## Step 7: Results table

Auto-built from everything logged via `log_experiment(...)` above.

In [ ]:
results_df = pd.DataFrame(results_log)
results_df

## Step 8: Final check — the test set

Every result so far is validation accuracy. Pick your best run from
the table above and check it against the true test set -- untouched
until now -- for one honest final number.

In [ ]:
best_model = baseline_model  # <-- replace with your actual best model once you have one
test_loss, test_accuracy = best_model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_accuracy:.4f}")

## Wrap-up

**Final question:** Looking at your results table and W&B dashboard,
which change moved validation accuracy the most, and which moved it
the least (or hurt it)? Reference specific run names.

_Write your answer here:_ `<FILL_IN>`